In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND SIMULATION CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
DIM_DATE = f"{CATALOG}.gold.dim_date"
DIM_EQUIPMENT = f"{CATALOG}.gold.dim_equipment"
SIMULATION_SCHEMA = "simulation"
LANDING = f"{CATALOG}.{SIMULATION_SCHEMA}.simulated_hourly_equipment_operations_landing"

SIMULATION_SEED = 20260819
SIMULATION_VERSION = "OEE_V1"
LOCAL_TIMEZONE = "Asia/Taipei"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SIMULATION_SCHEMA}")
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Day 4 deterministic hourly OEE configuration loaded.")


In [0]:
# ===================================================
# BLOCK 2 — CREATE THE COMPLETE HOUR/EQUIPMENT SPINE (PYTHON)
# ===================================================

date_df = spark.table(DIM_DATE).select(F.col("full_date").alias("production_date"))
equipment_df = spark.table(DIM_EQUIPMENT).select(
    "equipment_id", "site_id", "rated_units_per_hour"
)
hour_df = spark.range(24).select(F.col("id").cast("int").alias("production_hour_index"))

spine_df = (
    date_df.crossJoin(equipment_df).crossJoin(hour_df)
    .withColumn(
        "operation_hour_local",
        F.expr(
            "CAST(production_date AS TIMESTAMP) + "
            "make_interval(0, 0, 0, 0, production_hour_index + 8, 0, 0)"
        ),
    )
    .withColumn(
        "operation_hour_utc",
        F.to_utc_timestamp("operation_hour_local", LOCAL_TIMEZONE),
    )
)

expected_rows = 1826 * 24 * 24
actual_rows = spine_df.count()
assert actual_rows == expected_rows, (actual_rows, expected_rows)
print(f"Complete hour/equipment spine: {actual_rows:,} rows")



In [0]:
# ===================================================
# BLOCK 3 — GENERATE DETERMINISTIC TIME COMPONENTS (PYTHON)
# ===================================================

hashed_df = (
    spine_df
    .withColumn(
        "planned_hash",
        F.pmod(F.xxhash64(F.lit(SIMULATION_SEED), "equipment_id", "operation_hour_utc", F.lit("PLANNED")), 601),
    )
    .withColumn(
        "unplanned_hash",
        F.pmod(F.xxhash64(F.lit(SIMULATION_SEED), "equipment_id", "operation_hour_utc", F.lit("UNPLANNED")), 901),
    )
    .withColumn(
        "short_stop_hash",
        F.pmod(F.xxhash64(F.lit(SIMULATION_SEED), "equipment_id", "operation_hour_utc", F.lit("SHORT")), 301),
    )
    .withColumn(
        "setup_hash",
        F.pmod(F.xxhash64(F.lit(SIMULATION_SEED), "equipment_id", "operation_hour_utc", F.lit("SETUP")), 601),
    )
)

time_df = (
    hashed_df
    .withColumn("scheduled_time_seconds", F.lit(3600).cast("long"))
    .withColumn(
        "approved_planned_downtime_seconds",
        F.when(F.col("planned_hash") < 120, F.col("planned_hash")).otherwise(0).cast("long"),
    )
    .withColumn(
        "planned_production_time_seconds",
        F.col("scheduled_time_seconds") - F.col("approved_planned_downtime_seconds"),
    )
    .withColumn(
        "unplanned_downtime_seconds",
        F.least(
            F.when(F.col("unplanned_hash") < 300, F.col("unplanned_hash")).otherwise(0),
            F.col("planned_production_time_seconds"),
        ).cast("long"),
    )
    .withColumn(
        "operating_time_seconds",
        F.col("planned_production_time_seconds") - F.col("unplanned_downtime_seconds"),
    )
    .withColumn(
        "short_stop_seconds",
        F.least(F.col("short_stop_hash"), F.col("operating_time_seconds")).cast("long"),
    )
    .withColumn(
        "setup_seconds",
        F.least(
            F.col("setup_hash"),
            F.greatest(F.lit(0), F.col("operating_time_seconds") - F.col("short_stop_seconds")),
        ).cast("long"),
    )
    .withColumn("run_seconds", F.col("operating_time_seconds"))
)

In [0]:
# ===================================================
# BLOCK 4 — GENERATE OUTPUT AND QUALITY COMPONENTS (PYTHON)
# ===================================================

operations_df = (
    time_df
    .withColumn(
        "performance_basis_points",
        F.pmod(F.xxhash64(F.lit(SIMULATION_SEED), "equipment_id", "operation_hour_utc", F.lit("PERFORMANCE")), 2301) + 7500,
    )
    .withColumn(
        "quality_basis_points",
        F.pmod(F.xxhash64(F.lit(SIMULATION_SEED), "equipment_id", "operation_hour_utc", F.lit("QUALITY")), 681) + 9300,
    )
    .withColumn(
        "theoretical_output_units",
        F.col("operating_time_seconds").cast("double")
        * F.col("rated_units_per_hour").cast("double") / F.lit(3600.0),
    )
    .withColumn(
        "total_units",
        F.floor(F.col("theoretical_output_units") * F.col("performance_basis_points") / 10000.0).cast("long"),
    )
    .withColumn(
        "good_units",
        F.floor(F.col("total_units") * F.col("quality_basis_points") / 10000.0).cast("long"),
    )
    .withColumn(
        "operation_record_id",
        F.sha2(F.concat_ws("|", F.lit(SIMULATION_VERSION), F.lit(str(SIMULATION_SEED)), "equipment_id", F.col("operation_hour_utc").cast("string")), 256),
    )
    .select(
        "operation_record_id", "operation_hour_utc", "operation_hour_local",
        "production_date", "production_hour_index", "site_id", "equipment_id",
        "scheduled_time_seconds", "approved_planned_downtime_seconds",
        "planned_production_time_seconds", "unplanned_downtime_seconds",
        "operating_time_seconds", "short_stop_seconds", "setup_seconds",
        "run_seconds", "total_units", "good_units", "rated_units_per_hour",
        "theoretical_output_units",
        F.lit(SIMULATION_SEED).cast("long").alias("simulation_seed"),
        F.lit(SIMULATION_VERSION).alias("simulation_version"),
        F.lit(True).alias("simulated_record_flag"),
        F.lit("DETERMINISTIC_PORTFOLIO_SIMULATION").alias("record_origin"),
    )
)

In [0]:
signature_df = (
    operations_df
    .select(
        F.sha2(
            F.concat_ws(
                "|",
                "operation_record_id",
                "scheduled_time_seconds",
                "approved_planned_downtime_seconds",
                "unplanned_downtime_seconds",
                "total_units",
                "good_units",
                "simulation_seed",
            ),
            256,
        ).alias("row_signature"),
        F.col("total_units"),
        F.col("good_units"),
    )
    .agg(
        F.count("*").alias("row_count"),
        F.min("row_signature").alias("minimum_row_signature"),
        F.max("row_signature").alias("maximum_row_signature"),
        F.sum("total_units").alias("total_units"),
        F.sum("good_units").alias("good_units"),
    )
)

display(signature_df)